In [30]:
import os

file_path = "../dataset/AI_GENERATED_ABSTRACT_USING_GEMINI_REPHRASED.csv"
print("File exists:", os.path.exists(file_path))

File exists: True


In [32]:
import pandas as pd

ai_data = pd.read_csv("../dataset/AI_GENERATED_ABSTRACT_USING_GEMINI_REPHRASED.csv")
human_data = pd.read_csv("../dataset/abstract_dataset.csv")

In [2]:
human_data_df = pd.DataFrame(human_data)
ai_data_df = pd.DataFrame(ai_data)

In [4]:
concatenation_df = pd.concat([human_data, ai_data])

In [ ]:
concatenation_df.to_csv("../dataset/ai_human_written_abstract.csv")

In [8]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(concatenation_df.head())


                                         Author_name             topic  \
0                      Iain Carmichael, J. S. Marron  machine learning   
1  Jannis Kueck, Ye Luo, Martin Spindler, Zigan Wang  machine learning   
2                                          Jason Toy  machine learning   
3  Yu-Ren Liu, Yi-Qi Hu, Hong Qian, Chao Qian, Ya...  machine learning   
4                                  Abien Fred Agarap  machine learning   

                                               title  \
0         Data Science vs. Statistics: Two Cultures?   
1  Estimation and Inference of Treatment Effects ...   
2  SenseNet: 3D Objects Database and Tactile Simu...   
3    ZOOpt: Toolbox for Derivative-Free Optimization   
4  Towards Building an Intelligent Anti-Malware S...   

                                            Abstract  year Source  label  
0  Data science is the business of learning from ...  2017  arxiv  human  
1  Empirical researchers are increasingly faced w...  2017  arxiv  h

In [9]:
concatenation_df = concatenation_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(concatenation_df['Abstract'])

sequences = tokenizer.texts_to_sequences(concatenation_df['Abstract'])
max_length = 400  
X = pad_sequences(sequences, maxlen=max_length, padding='post')

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(concatenation_df['label'])  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
vocab_size = len(tokenizer.word_index) + 1  
embedding_dim = 128  
model_gru = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    GRU(64),  
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.summary()

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
epochs = 1
batch_size = 32

history = model_gru.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=1)


577/577 ━━━━━━━━━━━━━━━━━━━━ 178s 302ms/step - accuracy: 0.5071 - loss: 0.6941 - val_accuracy: 0.5161 - val_loss: 0.6928
